# 20.14 Kev的公開實作，哪些能轉成我們的小模型設計？


同一份客服訊息寫著「卡片還沒收到，申請編號是123」。我們想問兩題：這是卡片、地址還是程式問題？資料是否足夠處理？新版C已把問題變成有限候選上的機率（[19.23](https://birdhackor.github.io/tiny-perceptron-vlm/19.23.html)）。現在要問的是：成熟的決策模型如何安排相同工作，哪些設計值得借用，又有哪些能力不能直接算到我們身上？

Kev是Jared Palmer公開的Jev式獨立實作，提供相近的型別介面；它不是TypeSafe公布的Jev原始碼。其README引用了一篇外部架構分析，這也不等於Jev官方披露了訓練配方。本課原C依公開介面與訓練說明自行設計；此輪借用Kev可查的實作與測試，仍使用自己的小模型和資料。

Kev-4B把已訓練的Qwen3.5-4B-Base接上LoRA與pointer head。pointer head就是[9.17](https://birdhackor.github.io/tiny-perceptron-vlm/9.17.html)學過的候選指向式計分頭：每個選項的結尾token提供一條表示，題尾的decide token提供另一條，經兩個投影後算點積，再轉成機率。decide位於整串選項之後，所以能讀全部選項；某選項的表示也可能讀到前面的選項。這與本課獨立實質候選分支不同：交換選項可能改變Kev的表示與答案，不能僅憑計分頭用了點積，就推論重排一定不變。

兩個問題之間則希望互不影響。[16.15](https://birdhackor.github.io/tiny-perceptron-vlm/16.15.html)已區分「共用材料」與「共用所有可變狀態」。Kev-4B的32層底座有24層Gated DeltaNet、8層完整attention。這裡的DeltaNet層會把讀過的內容累積到循環狀態；分支要保護的不只有attention的KV cache，還有這些循環狀態及卷積狀態。只在attention上加遮罩，無法阻止另一條循環狀態帶入前一題。

下面用手寫小清單代表KV與循環狀態，觀察「從同一份背景開始」的意思。數字不是Kev的內部值，函式也不是神經推論。每題會加入一個KV項目，並更新一個循環數字；我們比較各題先完整複製背景，與只複製外層字典。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
from copy import deepcopy

prefix = {"kv": [7, 8], "recurrent": [10]}
questions = [("類別", 2), ("資訊足夠", 5)]


def branch(state, increment):
    state["kv"].append(increment)
    state["recurrent"][0] += increment
    return len(state["kv"]), state["recurrent"][0]


separate = [branch(deepcopy(prefix), n) for _, n in questions]
print("獨立分支", separate)
print("原背景", len(prefix["kv"]), prefix["recurrent"][0])

shared = dict(prefix)  # 內層兩個清單仍是同一份
contaminated = [branch(shared, n) for _, n in questions]
print("共用可變清單", contaminated)
print("被改過的背景", len(prefix["kv"]), prefix["recurrent"][0])

依序印出獨立分支 [(3, 12), (3, 15)]、原背景 2 10、共用可變清單 [(3, 12), (4, 17)]、被改過的背景 4 17。第一條路讓第二題從10開始加5；第二條路讓它接著第一題的12再加5，連KV長度都多了一格。真正Kev的cache是張量與專用類別，需要依其實作分出全部狀態；這個字典例只顯示漏掉可變內層會造成什麼問題，不證明已在本機驗過Kev。其一般Torch短列評測會逐題重算材料，服務路徑會共用prefix；兩條路應以相同權重、位置及精度另核對。

成熟底座也帶來本課沒有的知識與限制。此版4B模型卡主要語言是英語；服務接受最多65,536個材料token，不代表這個長度仍保有短文準確率。卡片依配對長文測試所列的驗證長度是8,192。其溫度T=2.41也不是我們中文任務的可靠性保證。這些分別回答語言、容納量、有效使用與校準問題，不能合成一句「能讀長文，所以一定會判」。

重現時還要固定adapter、底座、tokenizer、head與程式。這版adapter_config的revision是空值，training_config另記底座SHA；只固定adapter名稱仍可能載到另一版底座。來源中的完整pin與授權表見[本輪來源紀錄](https://github.com/birdhackor/tiny-perceptron-vlm/blob/83630ea1cea23d47c4b77162a31cbb7566686d11/docs/course-revision-20261029-omni/SOURCES.md)。

練習把第二題的增量5改成1。先預測獨立分支的第二個結果是(3, 11)，共用可變清單則是(4, 13)，再重跑核對。差別來自背景是否遭前一題改寫，並不是第二題比較難。

原始依據：[Kev固定碼版](https://github.com/jaredpalmer/kev/blob/2074cc44256200bb37540103fa4572901faa10f3/kev/model.py)的PointerHead與prefix分支；[4B模型卡](https://huggingface.co/jaredpalmer/kev-4b/blob/6cfce5c2fa4b4bd64026336ab649c5ca78857d52/README.md)的Model details、Long documents；[底座config](https://huggingface.co/Qwen/Qwen3.5-4B-Base/blob/1001bb4d826a52d1f399e183466143f4da7b741b/config.json)的layer_types。
